# G3b - four-cell concept screen (harm x refusal-geometry)

Finds concepts for the 2x2 that dissociates **semantic harmfulness** from **geometric refusal-alignment**:

|                | close to refusal (high cos) | far from refusal (low cos) |
|----------------|-----------------------------|----------------------------|
| **harmful**    | weapon, poison *(easy)*     | **harmful+distant** *(scarce)* |
| **harmless**   | grief, despair *(valence)*  | bread, orchid *(baseline)* |

**One execution screens all three Colab-sized models** (gemma-2-2b, gemma-3-4b, Qwen2.5-3B) back to back, freeing GPU between each - start it and walk away (~30-45 min total). Each model is wrapped in try/except, so a gated/failed model is skipped and the rest still complete.

**Two independent parts:**
1. **Candidate enumeration** - one shared, curated word list (identical across models, so geometry is comparable across families). Optionally augmented once from Gemma-Scope-2 transcoder labels; the mined vocabulary is portable.
2. **Geometry screen** - builds the *actual injected* Lindsey vector per candidate and measures `cos(v, d_refusal)` at the injection layer, per model.

> ### The one rule that keeps this valid
> A transcoder/SAE feature direction is **not** the vector you inject. So the **harm axis** is a search heuristic (vet the shortlist), and the **geometry axis** (`cos`) is **re-computed on the Lindsey vector you actually inject** - never taken from a feature. Keep construction = Lindsey and the 38.2% baseline gate stays intact.

> 🔒 Inference-only: no judge, no generation, no abliteration. Outputs are `cos` scalars + word lists - **safe to persist to Drive or the repo's `private/` (gitignored). Never persist vectors/activations/generations anywhere, and keep per-concept detail out of the public repo/paper (aggregate for anything public).** (CLAUDE.md)

## 0. Setup

Needs **only an HF token** (model download) - no judge key. Set `APIHuggingFace` in Colab Secrets, and accept the Gemma license on the gemma-2-2b and gemma-3-4b model pages under that same account (Qwen is ungated). **Runtime -> Change runtime type -> T4 GPU.**

In [ ]:
import os
%cd /content
if not os.path.exists('introspection-mechanisms'):
    !git clone https://github.com/safety-research/introspection-mechanisms.git
%cd /content/introspection-mechanisms
!pip install -q accelerate

def get_secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    return os.environ.get(name)

_hf = get_secret('APIHuggingFace') or get_secret('HF_TOKEN')
if _hf: os.environ['HF_TOKEN'] = _hf
print('HF token set:', bool(os.environ.get('HF_TOKEN')))

## 1. Config + shared candidate list

The candidate list is built **once** and screened identically across all models. `harm` is a semantic tag for search (only the `harmful` category is True by design); vet the off-diagonal shortlist before committing.

In [ ]:
import sys, torch, numpy as np, pandas as pd, gc, re, json
from tqdm.auto import tqdm
sys.path.insert(0, 'src'); sys.path.insert(0, 'experiments')
from transformers import AutoModelForCausalLM, AutoTokenizer

# ---- models screened in one run (all fit a free T4 at bf16) ----
MODELS = ['google/gemma-2-2b-it', 'google/gemma-3-4b-it', 'Qwen/Qwen2.5-3B-Instruct']
LAYER_FRACTION = 0.6      # ~60% depth; the 27B target injects at L37/62
N_PAIRS        = 120      # harmful/harmless pairs for d_refusal
N_BASELINE     = 100      # baseline words subtracted when building concept vectors
TOPK           = 5        # refusal-subspace dimension (cone robustness)
LOAD_4BIT      = False    # only to fit a big model on a small card (shifts borderline cos - not for final selection)

try:
    from vector_utils import get_baseline_words
    BASELINE_WORDS = get_baseline_words(N_BASELINE)
except Exception as e:
    print('get_baseline_words import failed, inline fallback:', e)
    BASELINE_WORDS = ('time year people way day man thing woman life child world school state family '
        'student group country problem hand part place case week company system program question work '
        'government number night point home water room mother area money story fact month lot right study '
        'book eye job word business issue side kind head house service friend father power hour game line '
        'end member law car city community name').split()[:N_BASELINE]

# ---- curated seed pools (single English words; words are not sensitive artifacts) ----
POOLS = {
 'benign':        'bread orchid apple hammer table cloud river pencil garden blanket lamp spoon window carpet mountain bicycle kettle pillow notebook umbrella ladder basket curtain teapot'.split(),
 'valence':       'sadness grief failure disgust loneliness boredom regret shame misery despair sorrow gloom anxiety dread humiliation guilt envy frustration disappointment heartbreak melancholy anguish'.split(),
 'harm_adjacent': 'police surgery medicine soldier ambulance prison vaccine hospital court firefighter military security patrol autopsy forensics quarantine arrest verdict trial guard sentry'.split(),
 'harmful':       'weapon poison assault explosive gun knife bomb murder torture grenade missile venom arson massacre stabbing shooting cyanide anthrax warfare terror kidnapping strangulation'.split(),
}
HARM_CATEGORY = {'benign':False,'valence':False,'harm_adjacent':False,'harmful':True}

# ---- optional: augment once from Gemma-Scope-2 transcoder labels (Gemma-Scope models only) ----
MINE_LABELS  = False
LABEL_LAYERS = [22, 23, 24]; LABEL_WIDTH='16k'; LABEL_L0='small'
HARM_LEX = set(('weapon gun knife bomb explosive grenade poison toxin venom cyanide anthrax assault murder '
    'kill torture attack violence abuse rape terror arson stab shoot wound massacre molest kidnap hostage '
    'missile warfare bloodshed mutilate strangle lynch sabotage detonate slaughter homicide manslaughter').split())
STOP = set('the a an and or of to in on for with about into over that this these those from your you it is are '
    'features feature related concept concepts words word text tokens token model layer neuron activation'.split())

def _mine(tok):
    from pathlib import Path
    from huggingface_hub import hf_hub_download
    found = {}
    for L in LABEL_LAYERS:
        data = None
        fn = f'gemma_scope_2_27b_transcoder_all_layer{L}_{LABEL_WIDTH}_{LABEL_L0}_labels.json'
        for c in [Path(f'gemma-scope-2/feature_labels/{fn}'), Path(f'../gemma-scope-2/feature_labels/{fn}')]:
            if c.exists(): data = json.loads(c.read_text()); break
        if data is None:
            try: data = json.loads(Path(hf_hub_download('google/gemma-scope-2-27b-it', f'feature_labels/{fn}')).read_text())
            except Exception as e: print(f'  layer {L}: no labels ({e})'); continue
        for lab in data.values():
            for w in re.findall(r'[a-z]{3,}', str(lab).lower()):
                if w in STOP or w in found: continue
                if len(tok(' '+w, add_special_tokens=False)['input_ids']) == 1: found[w] = str(lab)
    return found

rows, seen = [], set()
for cat, words in POOLS.items():
    for w in words:
        if w in seen: continue
        seen.add(w); rows.append(dict(concept=w, category=cat, harm=HARM_CATEGORY[cat]))
if MINE_LABELS:
    _t = AutoTokenizer.from_pretrained(MODELS[0])
    for w, lab in _mine(_t).items():
        if w in seen: continue
        seen.add(w); rows.append(dict(concept=w, category='mined', harm=any(k in (w+' '+lab).lower() for k in HARM_LEX)))
CAND_BASE = pd.DataFrame(rows)
print(f'{len(CAND_BASE)} candidates  |  harmful-tagged: {int(CAND_BASE.harm.sum())}  |  categories:',
      dict(CAND_BASE.category.value_counts()))

## 2. The screen function  *(one model in, cos table out; frees GPU on exit)*

In [ ]:
def screen_model(model_name, cand_base):
    tok = AutoTokenizer.from_pretrained(model_name)
    kw = dict(torch_dtype=torch.bfloat16, device_map='cuda', output_hidden_states=True)
    if LOAD_4BIT:
        from transformers import BitsAndBytesConfig
        kw = dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),
                  device_map='cuda', output_hidden_states=True)
    model = AutoModelForCausalLM.from_pretrained(model_name, **kw).eval()
    tcfg = getattr(model.config, 'text_config', model.config)   # Gemma-3 nests it; others flat
    N_LAYERS = tcfg.num_hidden_layers
    INJ = int(N_LAYERS * LAYER_FRACTION)

    @torch.no_grad()
    def pooled(user_text):
        prompt = tok.apply_chat_template([{'role':'user','content':user_text}], tokenize=False, add_generation_prompt=True)
        enc = tok(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
        hs = model(**enc, output_hidden_states=True).hidden_states
        start = 1 if enc['input_ids'].shape[1] > 1 else 0
        last = torch.stack([h[0, -1] for h in hs]).float().cpu()
        mx   = torch.stack([h[0, start:].amax(0) for h in hs]).float().cpu()
        return last, mx

    # d_refusal + refusal subspace at the injection layer (Arditi diff-of-means)
    import refusal_prompts
    H  = torch.stack([pooled(p)[0][INJ] for p in tqdm(refusal_prompts.HARMFUL_PROMPTS[:N_PAIRS],  desc=f'{model_name.split("/")[-1]}: harmful ')])
    Ml = torch.stack([pooled(p)[0][INJ] for p in tqdm(refusal_prompts.HARMLESS_PROMPTS[:N_PAIRS], desc=f'{model_name.split("/")[-1]}: harmless')])
    d = H.mean(0) - Ml.mean(0); d = d / (d.norm() + 1e-8)
    Uk = torch.linalg.svd(H - Ml.mean(0), full_matrices=False).Vh[:TOPK].T

    # baseline mean (shared subtraction)
    bl, bm = [], []
    for w in tqdm(BASELINE_WORDS, desc='baseline'):
        l, m = pooled(f'Tell me about {w}'); bl.append(l[INJ]); bm.append(m[INJ])
    base_last, base_mx = torch.stack(bl).mean(0), torch.stack(bm).mean(0)

    def cos(v): return float(((v/(v.norm()+1e-8)) * d).sum())
    out = cand_base.copy()
    cl, cm, sp = [], [], []
    for w in tqdm(out.concept.tolist(), desc='concept vecs'):
        l, m = pooled(f'Tell me about {w}')
        vl = l[INJ] - base_last; vm = m[INJ] - base_mx
        cl.append(cos(vl)); cm.append(cos(vm))
        sp.append(float((vl @ Uk @ Uk.T).norm() / (vl.norm() + 1e-8)))
    out['model'] = model_name.split('/')[-1]; out['inj_layer'] = INJ; out['n_layers'] = N_LAYERS
    out['cos_last'] = cl; out['cos_maxpool'] = cm; out['subspace_frac'] = sp

    del model, tok, H, Ml; gc.collect(); torch.cuda.empty_cache()
    return out

## 3. Run all models, report, save  *(leave this running)*

In [ ]:
# ---- where to persist results (cos scalars + words ONLY - safe; NEVER persist vectors/activations/generations) ----
PERSIST_TO    = 'drive'                                        # 'drive' (Colab) | 'private' (local repo) | 'ephemeral'
DRIVE_DIR     = '/content/drive/MyDrive/emergent-introspection/cell_screen'
PRIVATE_DIR   = '../private/cell_screen'                       # when the Emergent-Introspection repo is the working tree; adjust to your path
EPHEMERAL_DIR = 'cell_screen_out'                             # session disk; lost on disconnect

def persist_dir():
    if PERSIST_TO == 'drive':
        from google.colab import drive; drive.mount('/content/drive'); return DRIVE_DIR
    return PRIVATE_DIR if PERSIST_TO == 'private' else EPHEMERAL_DIR

results = []
for mn in MODELS:
    print(f'\n{"="*70}\n  screening {mn}\n{"="*70}')
    try:
        results.append(screen_model(mn, CAND_BASE))
    except Exception as e:
        print(f'  !! SKIPPED {mn}: {type(e).__name__}: {e}')
        gc.collect(); torch.cuda.empty_cache()

def report(df):
    thr = float(df.cos_last.median())                       # per-model split (geometry is model-specific)
    df = df.assign(geom=np.where(df.cos_last >= thr, 'close', 'distant'))
    print(f'\n### {df.model.iloc[0]}  (inj L{df.inj_layer.iloc[0]}/{df.n_layers.iloc[0]}, split cos_last={thr:+.3f})')
    print(df.pivot_table(index='harm', columns='geom', values='concept', aggfunc='count', fill_value=0))
    for mask, title in [((df.harm)&(df.geom=='distant'), 'HARMFUL + DISTANT (scarce; empty => harm~cos collinear => lean on abliteration)'),
                        ((~df.harm)&(df.geom=='close'),  'HARMLESS + CLOSE  (valence/topic coupling to refusal)')]:
        sub = df[mask].sort_values('cos_last')
        print(f'  -- {title}  (n={len(sub)})')
        for _, r in sub.iterrows():
            print(f'     {r.concept:16s} cos_last={r.cos_last:+.3f} cos_max={r.cos_maxpool:+.3f} [{r.category}]')

if results:
    allr = pd.concat(results, ignore_index=True)
    for mn in allr.model.unique():
        report(allr[allr.model == mn])
    outdir = persist_dir(); os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, 'cell_assignments_all_models.csv')
    allr.to_csv(path, index=False)                   # scalars + words only; never write vectors/activations here
    print('\nsaved ->', path)
else:
    print('no models completed - check the HF token and Gemma license acceptance')

## Hardware & timing

**This notebook (all 3 small models) - Colab free T4 (16GB), bf16.** gemma-2-2b (~5GB), gemma-3-4b (~8GB, multimodal - handled), Qwen2.5-3B (~6GB); each loaded then freed. Per model ~600-700 inference-only forward passes -> **~30-45 min for all three**, download included. Leave it running.

**Gemma-3-27B - separate, rent an 80GB card.** bf16 weights ~54GB, so a 48GB/40GB card (incl. Colab Pro+ A100 40GB) can't hold it.
- **GPU:** A100 80GB (RunPod ~\$1.4-1.9/hr, value pick) or H100 80GB (~\$2-3/hr).
- **Time/cost:** compute ~3-5 min; dominated by the one-time ~54GB download+load (~15-25 min) -> **under an hour, ~\$1-2.** Cheapest GPU step in the project.
- **dtype:** screen at **bf16** - cell assignment depends on `cos`, which selects your concepts. `LOAD_4BIT=True` fits a 24GB card but shifts borderline concepts across the split; rough look only, not the final selection.

To run the 27B here: add its name to `MODELS` on the 80GB box (and flip `MINE_LABELS=True` to fold in transcoder-label vocabulary, which only exists for that model).